In [1]:
import os
import sys
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from omegaconf import OmegaConf
from hydra.utils import instantiate
os.environ["HYDRA_FULL_ERROR"] = "1"
sys.path.append('..')

df = pd.read_csv('../data/raw/lt_reality.csv')
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
y_train = df_train.pop('price')
y_test = df_test.pop('price')

In [2]:
pre_conf = OmegaConf.load('../configs/preprocess.yaml')
preprocessor = instantiate(pre_conf, _convert_="all")
preprocessor.set_output(transform="pandas")
df_pre = preprocessor.fit_transform(df_train)
# df_pre.columns = df_pre.columns.map(normalize_name)
joblib.dump(preprocessor, './temp/preprocessor.joblib')

['./temp/preprocessor.joblib']

In [3]:
feats_conf = OmegaConf.load('../configs/featurizer.yaml')
featurizer = instantiate(feats_conf, _convert_="all")
df_feats = featurizer.fit_transform(df_pre)
joblib.dump(featurizer, './temp/featurizer.joblib')

['./temp/featurizer.joblib']

In [4]:
model_conf = OmegaConf.load('../configs/model.yaml')
model = instantiate(model_conf, _convert_="all")
model.fit(df_feats, y_train)
joblib.dump(model, './temp/model.joblib')

['./temp/model.joblib']

In [5]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('featurizer', featurizer),
    ('model', model)
])


In [6]:
pipeline.predict(df_test)

array([2.00264200e+07, 3.80176100e+07, 1.55961800e+07, 3.08315250e+07,
       6.36067150e+07, 5.07952500e+07, 2.90995850e+07, 3.63881150e+07,
       2.61028100e+07, 1.08033050e+07, 4.11807000e+07, 8.75680150e+07,
       5.06014550e+07, 1.43653900e+07, 3.80122700e+07, 5.68935350e+07,
       2.86896650e+07, 4.03710450e+07, 2.79309800e+07, 1.75622000e+07,
       4.59600700e+07, 8.15549250e+07, 4.90403100e+07, 3.35101400e+07,
       1.81381300e+07, 1.62895550e+07, 6.12615450e+07, 1.89325350e+07,
       7.85440000e+06, 7.89056400e+07, 1.39889900e+07, 2.91537500e+07,
       6.05312250e+07, 8.82000000e+06, 4.29402500e+07, 2.41698050e+07,
       9.23948850e+07, 4.08746650e+07, 1.21830900e+07, 8.25458400e+07,
       6.09518600e+07, 4.07597450e+07, 1.23014950e+07, 2.02296050e+07,
       1.56864450e+07, 5.70074200e+07, 2.14305700e+07, 4.28309100e+07,
       4.63057450e+07, 3.65935650e+07, 3.07959050e+07, 4.17257950e+07,
       2.54624000e+07, 6.95331300e+07, 1.79022250e+07, 7.26026300e+07,
      

In [7]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('featurizer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['property_id','location','property_type',...,'distance_to_metro_km', 'view_score','date_listed']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[['onehot', OneHotEncoder..._output=False), ...], ['ordinal', OrdinalEncode... Furnished']]), ...], ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `tr

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///../mlflow.db")

mlflow.set_experiment("reproducible-ml-pipeline-trial")

with mlflow.start_run() as run:
    mlflow.log_param("preprocessor", pre_conf)
    mlflow.log_param("featurizer", feats_conf)
    mlflow.log_param("model", model_conf)
    mlflow.sklearn.log_model(pipeline, "model")
    mlflow.log_metric("train_score", model.score(df_feats, y_train))
    mlflow.log_metric("test_score", model.score(df_test_feats, y_test))

In [10]:
pipeline.score(df_test, y_test)

0.9873815760303224

In [11]:
from dvclive import Live
import mlflow

In [ ]:
with Live() as live, mlflow.start_run() as run:
    df_pre = preprocessor.transform(df_test)
    df_feats = featurizer.transform(df_pre)
    y_pred = model.predict(df_feats)
    live.log("y_pred", y_pred)
    live.log("y_test", y_test)
    live.log("mse", ((y_pred - y_test) ** 2).mean())